#### Libraries

In [ ]:
import gym
import numpy as np
import itertools
import math
import tensorflow as tf
import random
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from gym import Env
from gym.spaces import MultiDiscrete, Discrete, Box
from itertools import permutations

# Particular

### Environment




The whole environment is custom. The only general thing is the structure of having:


1.   _ init _
    *   action_space
    *   observation_space
2.   step
    *   Conditions for changing the state 
    *   Reward allocation
3.   render
    *   Visualization (n/a)
4.   reset
    *   Reset state

In [ ]:
class BraidEnv(Env):

  def __init__(self, braid_strands, braid_initial=np.zeros(2, dtype=np.int32)):
    length = len(braid_initial)
    possible_numbers = (braid_strands-1)*2
    # There are 4*length(*possible_numbers) moves
    self.action_space = MultiDiscrete([4,length-1])
    # Set the observation space as [-k,...,-k],...,[k,...,k]
    low=np.append(np.ones([len(braid_initial)], dtype=np.int32)*(1-braid_strands),np.zeros(15, dtype=np.int32))
    high=np.append(np.ones([len(braid_initial)], dtype=np.int32)*(braid_strands-1),np.zeros(15, dtype=np.int32))
    self.observation_space = Box(low, high, dtype=np.int32)
    # Initialize the braid as the initial tangled given braid
    self.state = braid_initial
    # Constant initial state
    self.initial_state = np.copy(braid_initial)
    # Time starts in zero
    self.time = 0

  def step(self, action, repeated): 
    # action is an array of 2 entries, [0] move & [1] where does it start
    x = self.state
    reward = -1
    if action[1] <= len(x)-2 and action[1] >= 0 and action[0] <= 4 and action[0] > 0:
      # Swap k & 0 or k & h (<k-1)  action = [1, n]
      if action[0] == 1 and ((x[action[1]+1] == 0 and x[action[1]+1] != x[action[1]]) or (abs(x[action[1]+1]) < abs(x[action[1]])-1)):
        x[action[1]+1], x[action[1]] = x[action[1]], x[action[1]+1]
      else: 
        # Swap 0 & k or or h (<k-1) & k   action = [2, n] 
        if action[0] == 2 and ((x[action[1]] == 0 and x[action[1]+1] != x[action[1]]) or (abs(x[action[1]+1])-1 > abs(x[action[1]]))):
          x[action[1]+1], x[action[1]] = x[action[1]], x[action[1]+1]
        else:
          # Replace with 0s    action = [3, n]              
          if action[0] == 3 and x[action[1]+1] == -x[action[1]] and x[action[1]] != 0:
            x[action[1]+1], x[action[1]] = 0, 0
          else:
            # Swap inside out & the other 3rd RM moves     action = [4, n]
            if action[0] == 4 and action[1] <= len(x)-3 and x[action[1]] != 0 and x[action[1]+1] != 0 and x[action[1]+2] != 0 and abs(x[action[1]+2]) == abs(x[action[1]]):
              if x[action[1]] == x[action[1]+2] and (x[action[1]] == x[action[1]+1]+1 or x[action[1]] == x[action[1]+1]-1):
                x[action[1]], x[action[1]+1], x[action[1]+2] = x[action[1]+1], x[action[1]], x[action[1]+1]
              else:
                if x[action[1]] == x[action[1]+1]+1:
                  x[action[1]], x[action[1]+1], x[action[1]+2] = x[action[1]+2]+1, x[action[1]], x[action[1]+1]
                else:
                  if x[action[1]+1] == x[action[1]+2]+1:
                    x[action[1]], x[action[1]+1], x[action[1]+2] = x[action[1]+1], x[action[1]+2], x[action[1]]-1
                  else:
                    if x[action[1]+1]+1 == x[action[1]+2]:
                      x[action[1]], x[action[1]+1], x[action[1]+2] = x[action[1]+1], x[action[1]+2], x[action[1]]+1
                    else:
                      if x[action[1]]+1 == x[action[1]+2]:
                        x[action[1]], x[action[1]+1], x[action[1]+2] = x[action[1]+2]+1, x[action[1]], x[action[1]+1]
    else:
      pass
    if repeated == 1:
      reward = -10
    #print(action)
    self.state = np.array(x, np.int32)
    self.time += 1 
    # Check if braid is untangled & calculate reward
    if np.all(self.state == 0):
      reward = 10000
      done = True
    else:
      done = False
    # Set placeholder for info
    info = {}
    # Return step information
    return self.state, reward, done, info

  def render(self):
    pass

  def reset(self):
    # Reset the braid
    self.state = np.copy(self.initial_state)
    self.time = 0
    # Return the state after reset (initial state)
    return self.state

### Initialize the model

We create the model as:


1.   Input layer
    *   Size: braid_length+4*(braid_length-1)
2.   Hidden layer
    *   Size: n_inputs*braid_strands
    *   Activation: Elu
3.   Output layer
    *   Size: 4*(braid_length-1)
    *   Activation: Softmax (array of p's)



In [ ]:
def create_model(n_inputs, n_outputs, braid_strands):
  model = Sequential()
  model.add(Dense(n_inputs*braid_strands, activation="elu", input_shape=[n_inputs]))
  model.add(Dense(n_outputs, activation="softmax"))
  return model

### Agent looks for legal moves

The agent gets an array of legal moves that comes from the same set of conditions as the environment.

In [ ]:
def append_legal_moves(state):
  state_length = len(state)
  x = state[0:state_length]
  legal_moves = np.zeros(4*(state_length-1), dtype=np.int32)
  index = 0
  for action_position in range(0,state_length-1):
    for action_type in range(1,5):
      # Swap k & 0 or k & h (<k-1)  action = [1, n]
      if action_type == 1 and ((x[action_position+1] == 0 and x[action_position+1] != x[action_position]) or (abs(x[action_position+1]) < abs(x[action_position])-1)):
        index = action_type-1+action_position*4
        legal_moves[index] = 1
      else:
        # Swap 0 & k or or h (<k-1) & k   action = [2, n] 
        if action_type == 2 and ((x[action_position] == 0 and x[action_position+1] != x[action_position]) or (abs(x[action_position+1])-1 > abs(x[action_position]))):
          index = action_type-1+action_position*4
          legal_moves[index] = 1
        else:
          # Replace with 0s    action = [3, n]                
          if action_type == 3 and x[action_position+1] == -x[action_position] and x[action_position] != 0:
            index = action_type-1+action_position*4
            legal_moves[index] = 1
          else:
            # Swap inside out & other 3rd RM moves     action = [4, n]       
            if action_type == 4 and action_position <= len(x)-3 and x[action_position] != 0 and x[action_position+1] != 0 and x[action_position+2] != 0 and abs(x[action_position+2]) == abs(x[action_position]):
              flag = 0
              if x[action_position] == x[action_position+2] and (x[action_position] == x[action_position+1]+1 or x[action_position] == x[action_position+1]-1):
                flag = 1
              if x[action_position] == x[action_position+1]+1:
                flag = 1
              if x[action_position+1] == x[action_position+2]+1:
                flag = 1
              if x[action_position+1]+1 == x[action_position+2]:
                flag = 1
              if x[action_position]+1 == x[action_position+2]:
                flag = 1
              if flag ==1:
                index = action_type-1+action_position*4
                legal_moves[index] = 1
  x = np.append(x,legal_moves)
  return x

### Play one step

The gerenalities in this funtion are the following:

*   Gradient tape
*   Comparing the selected move against a random move
*   Get the loss function between the target move and the output of the model
*   Execute the target move

Some other parts of this code can be used for other __softmax__ output layer agents. 



In [ ]:
def play_one_step(env, braid_length, obs, n_outputs, model, loss_fn, store_list):
  with tf.GradientTape() as tape: 
    # Get the legal moves
    obs_legal = append_legal_moves(obs)
    legal_moves = obs_legal[braid_length:len(obs_legal)]
    legal_index = [index for index in range(len(legal_moves)) if legal_moves[index] == 1]
    # Get the output array from the model
    array_proba = model(obs_legal[np.newaxis])
    array_proba_legal = tf.convert_to_tensor([array_proba.numpy()[0][index] for index in legal_index])
    # Get the best move out of the legal moves
    max_proba = max([array_proba.numpy()[0][index] for index in legal_index])
    max_proba_zero = 0
    if max_proba > 0:
      max_index = np.where(array_proba.numpy()[0] == max_proba)[0][0]
    else:
      max_proba_zero = 1
      if len(legal_index) == 1:
        max_index = legal_index[0]
      else:
        index_1 = tf.random.uniform([1,1], minval=0, maxval=len(legal_index)-1, dtype=tf.int32).numpy()[0][0]
        max_index = legal_index[index_1]
    # Get the randomness of the move
    max_random = 1/math.sqrt(n_outputs)
    random = tf.random.uniform([1,1], minval=0, maxval=0.25) > tf.convert_to_tensor(max_proba[np.newaxis])
    # Create the target vector with 0's and a single 1 
    if tf.cast(random, dtype=tf.int32).numpy() == 0 or max_proba_zero == 1:
      index = max_index
    else:
      if len(legal_index) == 1:
        index = legal_index[0]
      else:
        legal_index.remove(max_index)
        if len(legal_index) == 1:
          index = legal_index[0]
        else:
          index_1 = tf.random.uniform([1,1], minval=0, maxval=len(legal_index)-1, dtype=tf.int32).numpy()[0][0]
          index = legal_index[index_1]
    y_target = np.zeros(n_outputs, dtype=np.int32)
    y_target[index] = 1
    y_target = tf.convert_to_tensor(y_target[np.newaxis])
    # Get the loss between the target and the output
    loss = tf.reduce_mean(loss_fn(y_target, array_proba))
  grads = tape.gradient(loss, model.trainable_variables)
  # Transform the selected move into a 2 entry array action
  action_array = np.array([index%4+1,index//4], dtype=np.int32)
  # Repetition for penalty
  repeated = 0
  if [obs.tolist(), action_array.tolist()] in store_list:
    repeated = 1
  store_list.append([obs.tolist(),action_array.tolist()])
  # Execute the action
  obs, reward, done, info = env.step(action_array, repeated)
  return obs, reward, done, grads, action_array

### Random braid set creation

This code was made by Dr. Alexei Vernitski.

In [ ]:
def reduce_word_in_kei_group(w):
  reduced = True
  while(reduced):
      reduced = False
      for i in range(len(w)-1):
          if w[i] == w[i+1]:
              w.pop(i)
              w.pop(i)
              reduced = True
              break
    #this function does not return the word, but changes the word as a side effect

def apply_sigma_negative(i, words):
    (words[i], words[i+1]) = (words[i] + words[i+1] + words[i], words[i])
    #this function does not return words, but changes words as a side effect

def apply_sigma_positive(i, words):
    (words[i], words[i+1]) = (words[i+1], words[i+1] + words[i] + words[i+1])
    #this function does not return words, but changes words as a side effect

def braid_to_automorphisms(braid, words):
  for s in braid:
      if s > 0:
          apply_sigma_positive(s-1, words)
      elif s < 0:
          apply_sigma_negative(-s-1, words)
  for i in range(len(words)):
      reduce_word_in_kei_group(words[i])

def is_braid_trivial(braid, braid_strands):
  words = [[i] for i in range(1, braid_strands+1)]
  braid_to_automorphisms(braid, words)
  is_trivial = True
  for i in range(braid_strands):
      if words[i] == [i+1]:
          continue
      is_trivial = False
  return is_trivial

def convert_set_to_list(set):
  list_of_tuples = sorted(set)
  list_of_lists = []
  for item in list_of_tuples:
    list_of_lists.append(list(item))
  return list_of_lists

def create_braids(braid_strands, braid_len):
  set_of_braids = set()
  with open("trivial_braid_examples.txt", "w") as output_file:
    n = braid_strands - 1
    letters = list(range(-n, 0)) + list(range(1, n+1))
    how_many_braids_generated = 0
    while(how_many_braids_generated < 1000):
        braid = []
        for i in range(braid_len):
            braid.append(random.choice(letters))
        if tuple(braid) in set_of_braids: continue
        if is_braid_trivial(braid, braid_strands):
            how_many_braids_generated += 1
            print(braid, file=output_file)
            set_of_braids.add(tuple(braid))
  return convert_set_to_list(set_of_braids)


# General

### Play multiple episodes

In [ ]:
def play_multiple_episodes(env, braid_length, n_episodes, n_max_steps, n_outputs, model, loss_fn): 
  all_rewards = []
  all_grads = []
  all_steps = []
  all_wl = []
  # Loop for number of episodes
  for episode in range(n_episodes):
    current_rewards = [] 
    current_grads = []
    n_steps = 0
    win = 0
    obs = env.reset()
    last_action = np.zeros(15, dtype=np.int32)
    store_list = []
    # Loop for number of steps steps
    for step in range(n_max_steps):
      obs, reward, done, grads, action = play_one_step(env, braid_length, obs, n_outputs, model, loss_fn, store_list) 
      current_rewards.append(reward)
      current_grads.append(grads)
      last_action = np.copy(action)
      n_steps += 1
      if done:
        win = 1
        break
    all_rewards.append(current_rewards)
    all_grads.append(current_grads)
    all_steps.append(n_steps) 
    all_wl.append(win) 
  return all_rewards, all_grads, all_steps, all_wl

### Discount rewards

In [ ]:
def discount_rewards(rewards, discount_factor):
  discounted = np.array(rewards, dtype=np.float16)
  for step in range(len(rewards) - 2, -1, -1):
    discounted[step] += discounted[step + 1] * discount_factor
  return discounted

In [ ]:
def discount_all_rewards(all_rewards, discount_factor):
  all_discounted_rewards = [discount_rewards(rewards, discount_factor) for rewards in all_rewards]
  return [discounted_rewards for discounted_rewards in all_discounted_rewards]

### Neural network training

In [ ]:
def train_network(env, model, braid, iteration, n_episodes_to_update, n_max_steps, n_outputs, discount_factor, optimizer, loss_fn, total_wl):
  all_rewards, all_grads, all_steps, all_wl = play_multiple_episodes(env, len(braid), n_episodes_to_update, n_max_steps, n_outputs, model, loss_fn)
  total_mean_rewards = sum(map(sum, all_rewards)) / n_episodes_to_update
  total_mean_steps = sum(all_steps) / n_episodes_to_update     
  total_wins = sum(all_wl)             
  print("\r{}.- Braid: {}, Mean rewards: {:.1f}, Mean steps: {}, Final state: {}, Wins: {}".format(iteration, braid, total_mean_rewards, total_mean_steps, env.state, total_wins), end="")
  print('')
  all_final_rewards = discount_all_rewards(all_rewards,discount_factor)
  all_mean_grads = []
  for var_index in range(len(model.trainable_variables)):
    mean_grads = tf.reduce_mean([final_reward * all_grads[episode_index][step][var_index]
    for episode_index, final_rewards in enumerate(all_final_rewards) 
      for step, final_reward in enumerate(final_rewards)], axis=0)
    all_mean_grads.append(mean_grads)
  optimizer.apply_gradients(zip(all_mean_grads, model.trainable_variables))
  total_wl.append(total_wins)
  return total_wl

# Main

### Main function
Loops on "iterations"

In [ ]:
def start_training(model,braid_set,n_iterations,n_episodes_to_update,n_max_steps,n_outputs,discount_factor, braid_strands, braid_length):
  optimizer = keras.optimizers.Adam(learning_rate=0.02)
  loss_fn = tf.keras.losses.BinaryCrossentropy()
  training_set = braid_set #create_braids(braid_strands, braid_length) #create_braids_0(initial_state) # [[1,0,0,0,0,-1]] # [[-2, -1, -3, 1, 3, 2]]#
  total_wl = []
  for iteration in range(n_iterations):  
    braid = training_set[random.randint(0,len(training_set)-1)]
    env = BraidEnv(braid_strands,np.array(braid, np.int32))
    total_wl = train_network(env, model, braid, iteration, n_episodes_update, n_max_steps, n_outputs, discount_factor, optimizer, loss_fn, total_wl)
    env.close()
  return

### Hyperparameters

In [ ]:
n_iterations = 1000
n_episodes_update = 2
n_max_steps = 100
discount_factor = 0.95
braid_strands = 4
braid_length = 10
n_inputs = braid_length+4*(braid_length-1)
n_outputs = 4*(braid_length-1)

### Running the funtions

#### Initialization of the training set and the model

In [ ]:
braid_set = create_braids(braid_strands, braid_length)

In [ ]:
model = create_model(n_inputs, n_outputs, braid_strands)

#### Training the agent

In [ ]:
start_training(model, braid_set, n_iterations, n_episodes_update, n_max_steps, n_outputs, discount_factor, braid_strands, braid_length)

#### Saving the agent

In [ ]:
model.save('/Users/mateosallesize/Google Drive/Colab Notebooks/Braids')

INFO:tensorflow:Assets written to: /Users/mateosallesize/Google Drive/Colab Notebooks/Braids/assets


In [ ]:
model.cool = keras.models.load_model('/Users/mateosallesize/Google Drive/Colab Notebooks/Braids')